# A1-G08 — Customer Segmentation with RFM + K-Means

**Full pipeline:** Data Loading & Understanding → RFM Feature Engineering → Scaling → K Selection → K-Means → PCA Visualization → Cluster Interpretation → Personas & Recommendations.

This notebook merges the group's two prepared parts and completes the remaining sections (PCA, cluster interpretation, personas, business recommendations).

## 1. Data Loading & Understanding

We start from the transaction-level dataset and inspect its structure before creating customer-level RFM features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset is already in the same project folder as this notebook.
df = pd.read_csv("retail_transactions_segmentation.csv")

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# Basic structure and data types
df.info()

In [ ]:
# Descriptive statistics
display(df.describe(include="all"))

In [ ]:
# Missing values
missing_values = df.isnull().sum()
display(missing_values[missing_values > 0])

# Duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
# Convert transaction_date to datetime
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

print("Date range:")
print("Minimum:", df["transaction_date"].min())
print("Maximum:", df["transaction_date"].max())

print("\nData types after conversion:")
print(df.dtypes)

## 2. RFM Feature Engineering

RFM summarizes customer behavior using three features:

- **Recency:** number of days since the customer's most recent purchase.
- **Frequency:** number of transactions made by the customer.
- **Monetary:** total amount spent by the customer.

For this project, Monetary is defined as **total spend per customer**.

The reference date is defined explicitly as **one day after the latest transaction date in the dataset**. This makes the latest customer's Recency equal to 1 rather than 0.

In [ ]:
# Fixed reference date for Recency
reference_date = df["transaction_date"].max() + pd.Timedelta(days=1)

print("Reference date:", reference_date.date())

In [ ]:
# Build the customer-level RFM table
rfm = (
    df.groupby("customer_id")
      .agg(
          Recency=("transaction_date",
                   lambda x: (reference_date - x.max()).days),
          Frequency=("transaction_date", "count"),
          Monetary=("amount", "sum")
      )
      .reset_index()
)

# Round Monetary for readability
rfm["Monetary"] = rfm["Monetary"].round(2)

display(rfm.head())
print("RFM table shape:", rfm.shape)

In [ ]:
# Verify that every customer appears once and the RFM table is complete
print("Unique customers:", rfm["customer_id"].nunique())
print("\nMissing values in RFM:")
display(rfm.isnull().sum())

print("\nRFM descriptive statistics:")
display(rfm[["Recency", "Frequency", "Monetary"]].describe())

## 3. Basic RFM Distributions

These plots provide a quick view of the distributions before the next team member performs scaling and clustering.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(rfm["Recency"], kde=True, ax=axes[0])
axes[0].set_title("Recency Distribution")
axes[0].set_xlabel("Recency (days)")

sns.histplot(rfm["Frequency"], kde=True, ax=axes[1])
axes[1].set_title("Frequency Distribution")
axes[1].set_xlabel("Number of Transactions")

sns.histplot(rfm["Monetary"], kde=True, ax=axes[2])
axes[2].set_title("Monetary Distribution")
axes[2].set_xlabel("Total Spend")

plt.tight_layout()
plt.show()

## Handoff to the Next Team Member

The output of this section is the `rfm` DataFrame with one row per customer and the three features:

`Recency`, `Frequency`, and `Monetary`.

The next section of the group notebook should standardize these three RFM features before K-Means clustering.

## 1. Prepare RFM Features for Clustering

K-Means is distance-based, so the three RFM variables are standardized before clustering.

In [ ]:
from sklearn.preprocessing import StandardScaler

rfm_features = ["Recency", "Frequency", "Monetary"]

X_rfm = rfm[rfm_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=rfm_features,
    index=rfm.index
)

display(X_scaled_df.head())

## 2. Elbow Method and Silhouette Score

We evaluate several values of K. The Elbow Method uses inertia, while the Silhouette Score measures how well-separated the clusters are.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K_range = range(2, 11)
inertia_list = []
silhouette_list = []

for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = kmeans.fit_predict(X_scaled)

    inertia_list.append(kmeans.inertia_)
    silhouette_list.append(silhouette_score(X_scaled, labels))

results = pd.DataFrame({
    "K": list(K_range),
    "Inertia": inertia_list,
    "Silhouette Score": silhouette_list
})

display(results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(K_range), inertia_list, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("Inertia")
axes[0].set_xticks(list(K_range))
axes[0].grid(True)

axes[1].plot(list(K_range), silhouette_list, marker="o")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_xticks(list(K_range))
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 3. Select K

We select the K value using both the Elbow Method and Silhouette Score.

The code below reports the K with the highest Silhouette Score as a data-driven candidate. The final choice should also consider the Elbow plot and business interpretability.

In [ ]:
best_silhouette_k = int(
    results.loc[results["Silhouette Score"].idxmax(), "K"]
)

best_silhouette_score = results["Silhouette Score"].max()

print("K with highest Silhouette Score:", best_silhouette_k)
print("Highest Silhouette Score:", round(best_silhouette_score, 4))

In [ ]:
# Final K used for the K-Means model.
# Change this value only after the team reviews the Elbow and Silhouette plots.
selected_k = best_silhouette_k

print("Selected K:", selected_k)

## 4. Train the Final K-Means Model

The final K-Means model uses the selected K and `random_state=42` for reproducibility.

In [ ]:
kmeans_final = KMeans(
    n_clusters=selected_k,
    random_state=42,
    n_init=10
)

rfm["Cluster"] = kmeans_final.fit_predict(X_scaled)

display(rfm.head())

In [ ]:
# Check cluster sizes
cluster_counts = (
    rfm["Cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("Cluster")
    .reset_index(name="Customer Count")
)

display(cluster_counts)

print("Total customers:", len(rfm))
print("Total assigned to clusters:", rfm["Cluster"].notna().sum())

In [ ]:
# RFM profile by cluster
cluster_profile = (
    rfm.groupby("Cluster")[rfm_features]
       .mean()
       .round(2)
)

display(cluster_profile)

## Handoff to the Next Team Member

At this point, the `rfm` DataFrame contains the final `Cluster` label for every customer.

The next section should use this same `rfm` DataFrame to:
1. Apply PCA to the scaled RFM features.
2. Visualize the clusters in 2D.
3. Interpret each cluster.
4. Create customer personas and business recommendations.

## 5. PCA for 2D Cluster Visualization

The three standardized RFM features are reduced to 2 principal components so the clusters found by K-Means can be visualized on a single scatter plot.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

rfm["PC1"] = X_pca[:, 0]
rfm["PC2"] = X_pca[:, 1]

print("Explained variance ratio per component:", pca.explained_variance_ratio_.round(3))
print("Total variance explained by 2 components:",
      round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=rfm,
    x="PC1", y="PC2",
    hue="Cluster",
    palette="tab10",
    s=60,
    alpha=0.85
)
plt.title(f"Customer Segments in 2D (PCA) — K={selected_k}")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Cluster Interpretation (RFM Scoring)

Each RFM feature is converted into a quartile score from 1 (worst) to 4 (best), so clusters can be interpreted consistently regardless of the raw scale of Recency, Frequency, and Monetary.

- **Recency:** lower raw value is better → reversed so 4 = most recent.
- **Frequency:** higher raw value is better → 4 = most frequent.
- **Monetary:** higher raw value is better → 4 = highest spend.

In [ ]:
# Quartile scores (1 = worst, 4 = best) for each RFM dimension
rfm["R_Score"] = pd.qcut(rfm["Recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 4, labels=[1, 2, 3, 4]).astype(int)

rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

cluster_scores = (
    rfm.groupby("Cluster")[["R_Score", "F_Score", "M_Score", "RFM_Score"]]
       .mean()
       .round(2)
)

display(cluster_scores)

In [ ]:
# Assign a human-readable segment name to each cluster based on its average
# RFM scores relative to the overall (across-cluster) average.
def label_segment(row, r_mid, f_mid, m_mid):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]
    if r >= r_mid and f >= f_mid and m >= m_mid:
        return "Champions"
    elif r >= r_mid and f < f_mid and m < m_mid:
        return "New / Promising Customers"
    elif r < r_mid and f >= f_mid and m >= m_mid:
        return "At Risk (High Value)"
    elif r < r_mid and f < f_mid and m < m_mid:
        return "Lost / Churned"
    else:
        return "Regular Customers"

r_mid = cluster_scores["R_Score"].mean()
f_mid = cluster_scores["F_Score"].mean()
m_mid = cluster_scores["M_Score"].mean()

cluster_scores["Segment"] = cluster_scores.apply(
    lambda row: label_segment(row, r_mid, f_mid, m_mid), axis=1
)

display(cluster_scores)

## 7. Customer Personas & Business Recommendations

Each cluster is mapped to a marketing-friendly persona with a concrete recommendation, using the average RFM scores computed above.

In [ ]:
persona_recommendations = {
    "Champions": "Reward loyalty (VIP perks, early access to new products), ask for reviews/referrals, and upsell premium items — this is the most valuable, most engaged group.",
    "New / Promising Customers": "Recent buyers who haven't purchased often or spent much yet. Onboard with welcome offers and encourage a fast second purchase to build the habit.",
    "At Risk (High Value)": "Historically valuable and frequent, but haven't purchased recently. Prioritize win-back campaigns and personalized discounts before they churn completely.",
    "Lost / Churned": "Low recency, frequency, and spend. Use low-cost reactivation emails, or deprioritize marketing spend on this group if reactivation cost outweighs expected return.",
    "Regular Customers": "Steady but not exceptional across R, F, M. Nurture with regular engagement, cross-selling, and a loyalty program to push them toward becoming Champions.",
}

for cluster_id, row in cluster_scores.iterrows():
    segment = row["Segment"]
    print(f"Cluster {cluster_id} -> {segment}")
    print(f"  Avg R Score: {row['R_Score']} | Avg F Score: {row['F_Score']} | Avg M Score: {row['M_Score']} | Avg RFM Score: {row['RFM_Score']}")
    print(f"  Recommendation: {persona_recommendations.get(segment, 'Review this cluster manually.')}")
    print()

In [ ]:
# Attach the segment label back onto every customer
segment_map = cluster_scores["Segment"].to_dict()
rfm["Segment"] = rfm["Cluster"].map(segment_map)

display(rfm[["customer_id", "Recency", "Frequency", "Monetary", "Cluster", "Segment"]].head(10))

print("\nCustomer count per segment:")
display(rfm["Segment"].value_counts())

In [ ]:
# Visual summary: segment sizes and average spend per segment
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

segment_counts = rfm["Segment"].value_counts()
axes[0].bar(segment_counts.index, segment_counts.values, color=sns.color_palette("tab10"))
axes[0].set_title("Customer Count by Segment")
axes[0].set_ylabel("Number of Customers")
axes[0].tick_params(axis="x", rotation=30)

segment_monetary = rfm.groupby("Segment")["Monetary"].mean().sort_values(ascending=False)
axes[1].bar(segment_monetary.index, segment_monetary.values, color=sns.color_palette("tab10"))
axes[1].set_title("Average Monetary Value by Segment")
axes[1].set_ylabel("Average Total Spend")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 8. Final Summary

- Built customer-level **RFM** features (Recency, Frequency, Monetary) from the raw transaction data.
- **Standardized** the features and used the **Elbow Method** and **Silhouette Score** to select the number of clusters `K`.
- Trained the final **K-Means** model and assigned each customer to a cluster.
- Used **PCA** to visualize the clusters in 2D.
- Scored each cluster on Recency/Frequency/Monetary and translated the clusters into **business-readable personas** (Champions, At Risk, Lost/Churned, New/Promising, Regular) with concrete **marketing recommendations** for each.

The final `rfm` DataFrame now contains, per customer: `customer_id`, `Recency`, `Frequency`, `Monetary`, `Cluster`, `PC1`, `PC2`, `R_Score`, `F_Score`, `M_Score`, `RFM_Score`, and `Segment` — ready for reporting or a slide deck.